# Fine-Tune a Robot Policy with LoRA

In this hands-on workshop, adapt the official LingBot-VLA-v2 checkpoint to RoboTwin click_alarmclock on one AMD ROCm GPU. Learn from 50 clean task demonstrations, then compare the base and adapted policies on the same 10 clean and 10 randomized episodes.

Key takeaways:

- Adapt a pretrained robot policy
- Learn from task demonstrations
- Check closed-loop task success

Run the notebook inside the prepared RoboTwin image. A paired failure-to-success video example is available in a final optional cell, off by default.


## Before we begin: the robot policy

LingBot-VLA-v2 is a vision-language-action policy. It takes camera observations and a task instruction, then predicts robot actions. RoboTwin closes the loop: observe the scene, act, and observe again.

We begin with the official pretrained checkpoint. A seed fixes one evaluation episode, so the base and trained policies can be compared on the same 10 clean and 10 randomized episodes.


## Two ways to adapt the policy

| Approach | What learns during training |
|---|---|
| LoRA | Small added weights in the attention layers; the pretrained model weights stay frozen. |
| Full-parameter SFT | All trainable model weights are updated from demonstrations. |

This notebook is the LoRA session. Its compact adapter is trained on one ROCm GPU; the separate SFT notebook demonstrates the full-parameter approach on four ROCm GPUs.


## 0. Discover one idle ROCm GPU

The notebook selects an idle GPU automatically and prints its model and architecture. This same GPU runs LoRA training and both clean/randomized evaluations. The adapter recipe remains micro batch 4, gradient accumulation 2 and global batch 8.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, time

PROJECT_ROOT = Path('/RoboTwin')
SCRIPT_ROOT = PROJECT_ROOT / 'experiments/lingbot_vla_v2_6b_robotwin/scripts'
EXP = PROJECT_ROOT / 'experiments/lingbot_vla_v2_6b_robotwin'
SOURCE = EXP / 'source/lingbot-vla-v2'
BASE = EXP / 'models/robbyant_lingbot-vla-v2-6b'
YAML = EXP / 'training/lingbotvla_cli.yaml'
assert all((SCRIPT_ROOT / name).is_file() for name in
           ('eval/run_paired_eval.py', 'eval/select_idle_gpus.py')), 'Workshop scripts are missing from the image.'
sys.path.insert(0, str(SCRIPT_ROOT / 'eval'))

GPU_SELECTION = json.loads(subprocess.check_output(
    [sys.executable, str(SCRIPT_ROOT / 'eval/select_idle_gpus.py'), '--count', '1'],
    text=True))
GPU_IDS = GPU_SELECTION['gpu_ids']
GPU_CARDS = GPU_SELECTION['gpu_cards']
GPU_ID = GPU_IDS[0]
EVAL_GPU = GPU_ID
TASK_NAME = 'click_alarmclock'
SEEDS = [100000, 100003, 100004, 100007, 100009,
         100011, 100012, 100013, 100014, 100015]
PORT = 13690
STEPS = 25
MICRO_BATCH = 4
GRAD_ACCUM = 2
GLOBAL_BATCH = len(GPU_IDS) * MICRO_BATCH * GRAD_ACCUM
GPU_LABEL = '-'.join(map(str, GPU_IDS))
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
RUN_NAME = f'workshop_{TASK_NAME}_gpu{GPU_ID}_{stamp}'
RUN_DIR = Path('/workspace/runtime/outputs/robotwin-lora-workshop-runs') / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
DATA = PROJECT_ROOT / f'data/lerobot/{TASK_NAME}_joint_v30'
POLICY_SOURCE = SOURCE
EVALUATOR = SCRIPT_ROOT / 'eval/paired_seed_evaluator.py'
TRAIN_SOURCE = RUN_DIR / 'source_lora'
CHECKPOINT = RUN_DIR / f'checkpoints/global_step_{STEPS}/model'
ADAPTER = RUN_DIR / 'adapter.safetensors'
MERGED = RUN_DIR / f'merged_checkpoint/global_step_{STEPS}/hf_ckpt_streamed'
TIMING_PATH = RUN_DIR / 'stage_times.json'
stage_times = json.loads(TIMING_PATH.read_text()) if TIMING_PATH.exists() else {}
notebook_started = time.monotonic()

def save_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n')

def record_time(stage, started):
    stage_times[stage] = {'seconds': round(time.monotonic() - started, 1)}
    save_json(TIMING_PATH, stage_times)

def run_logged(command, log, *, env=None, cwd=SCRIPT_ROOT):
    with log.open('w') as output:
        result = subprocess.run(command, cwd=cwd, env=env,
                                stdout=output, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(f'Command failed (rc={result.returncode}); see {log}')

def evaluate(method):
    started = time.monotonic()
    command = [sys.executable, str(SCRIPT_ROOT / 'eval/run_paired_eval.py'),
               '--run-dir', str(RUN_DIR), '--result-tag', 'no_video',
               '--hip-gpu', str(EVAL_GPU), '--port', str(PORT),
               '--count', str(len(SEEDS)), '--steps', str(STEPS),
               '--task-name', TASK_NAME, '--method', method]
    run_logged(command, RUN_DIR / f'evaluate_{method}_no_video.log')
    record_time(f'evaluate_{method}_no_video', started)
    return json.loads((RUN_DIR / 'evaluations/no_video/summary.json').read_text())

print('run:', RUN_DIR)
print('task:', TASK_NAME, 'seeds:', SEEDS)
print('training/evaluation GPU:', GPU_ID, 'micro:', MICRO_BATCH,
      'accum:', GRAD_ACCUM, 'global:', GLOBAL_BATCH)
for gpu in GPU_IDS:
    card = GPU_CARDS[str(gpu)]
    print(f"GPU {gpu}: {card['model']} ({card['arch']})")


## 1. Check the checkpoint, data and GPU

Verify the official checkpoint and training YAML hashes, the 50 clean task demonstrations and the shared 10-seed list. Recheck that the automatically selected GPU is idle and keep its render-device mapping. Save a manifest before running either policy.


In [ ]:
started = time.monotonic()
if not TASK_NAME.replace('_', '').isalnum():
    raise ValueError('Invalid task name')
for path in (BASE / 'model.safetensors.index.json', DATA / 'meta/info.json', YAML,
             SOURCE, SCRIPT_ROOT / 'training/run_lora_task.sh',
             SCRIPT_ROOT / 'checkpoints/extract_lora_adapters_dcp.py',
             SCRIPT_ROOT / 'checkpoints/merge_lora_adapters_streaming.py'):
    if not path.exists():
        raise FileNotFoundError(path)
if hashlib.sha256((BASE / 'model.safetensors.index.json').read_bytes()).hexdigest() != '5a753ec331c51925d064e1e76e921a7eb3dca9770d2a6a91dac5e0e4162d676a':
    raise RuntimeError('Unexpected official base checkpoint index')
if hashlib.sha256(YAML.read_bytes()).hexdigest() != '808a2718c81595e0303fa8b5d4b563d004014395711ae9cf371acd1a0ab635df':
    raise RuntimeError('Unexpected training YAML')
info = json.loads((DATA / 'meta/info.json').read_text())
if info['total_episodes'] != 50:
    raise RuntimeError(f'Expected 50 clean {TASK_NAME} demonstrations')
if len(SEEDS) != 10 or len(SEEDS) != len(set(SEEDS)):
    raise RuntimeError('Expected exactly 10 unique seeds in each scene')

memory = json.loads(subprocess.check_output(['rocm-smi', '--showmeminfo', 'vram', '--json'], text=True))
cards = {}
for gpu in GPU_IDS:
    details = dict(GPU_CARDS[str(gpu)])
    used = int(memory[details['card']]['VRAM Total Used Memory (B)']) / 2**30
    if used >= 1:
        raise RuntimeError(f"Selected GPU {gpu} became busy: {used:.2f} GiB used")
    details['vram_used_gib'] = round(used, 3)
    cards[str(gpu)] = details
import torch, triton, sapien
manifest = {
    'mode': 'single',
    'run_name': RUN_NAME, 'gpu_ids': GPU_IDS, 'eval_gpu': EVAL_GPU,
    'gpu_cards': cards, 'task': TASK_NAME,
    'dataset_episodes': info['total_episodes'], 'dataset_frames': info['total_frames'],
    'training_seed': 42, 'base_index_sha256': '5a753ec331c51925d064e1e76e921a7eb3dca9770d2a6a91dac5e0e4162d676a',
    'training_yaml_sha256': '808a2718c81595e0303fa8b5d4b563d004014395711ae9cf371acd1a0ab635df',
    'micro_batch': MICRO_BATCH, 'gradient_accumulation': GRAD_ACCUM,
    'global_batch': GLOBAL_BATCH, 'steps': STEPS,
    'method': 'lora', 'lora_rank': 16, 'lora_alpha': 32,
    'learning_rate': 5e-4,
    'seeds': SEEDS, 'torch': torch.__version__, 'hip': torch.version.hip,
    'triton': triton.__version__, 'sapien': sapien.__version__,
}
save_json(RUN_DIR / 'manifest.json', manifest)
save_json(RUN_DIR / 'paired_eval_seeds.json', SEEDS)
(RUN_DIR / 'task_only.txt').write_text(f'robotwin {DATA}\n')
record_time('preflight', started)
print('dataset:', info['total_episodes'], 'demonstrations,', info['total_frames'], 'frames')


## 2. Check the shared evaluation implementation

The image already contains the per-episode policy seed patch and the workshop-only MoE fallback. Both policies use the same evaluator and the same source tree; the notebook verifies their hashes before evaluation.


In [ ]:
started = time.monotonic()
policy_file = POLICY_SOURCE / 'deploy/lingbot_vla_v2_policy.py'
for path in (policy_file, EVALUATOR):
    if not path.is_file():
        raise FileNotFoundError(path)
if 'ROBOTWIN_WORKSHOP_DISABLE_FUSED_MOE' not in policy_file.read_text():
    raise RuntimeError('Workshop policy patch is missing; rebuild the image')
if '"seed": int(now_seed)' not in EVALUATOR.read_text():
    raise RuntimeError('Paired evaluator lacks per-episode policy seeds')
manifest = json.loads((RUN_DIR / 'manifest.json').read_text())
manifest['policy_sha256'] = hashlib.sha256(policy_file.read_bytes()).hexdigest()
manifest['evaluator_sha256'] = hashlib.sha256(EVALUATOR.read_bytes()).hexdigest()
save_json(RUN_DIR / 'manifest.json', manifest)
record_time('prepare_evaluation', started)
print('policy:', POLICY_SOURCE)
print('evaluator:', EVALUATOR)


## 3. Evaluate the base policy

Evaluate the official base checkpoint on the same 10 clean and 10 randomized seeds. This main evaluation records success and failure for each episode without video.


In [ ]:
baseline_results = evaluate('baseline')
print('baseline:', baseline_results)


## 4. Train task-specific LoRA

Make an isolated source copy, then train only on the task's 50 clean demonstrations. Use rank 16, alpha 32, Q/K/V/O attention targets, learning rate 5e-4, seed 42 and 25 optimizer steps on the selected ROCm GPU.


In [ ]:
log = RUN_DIR / 'train.log'
if CHECKPOINT.is_dir() and log.is_file() and f'Reached max_steps={STEPS}' in log.read_text(errors='replace'):
    print('Using completed LoRA checkpoint:', CHECKPOINT)
else:
    if CHECKPOINT.exists():
        raise RuntimeError(f'Incomplete LoRA checkpoint: {CHECKPOINT}')
    started = time.monotonic()
    if not TRAIN_SOURCE.is_dir():
        shutil.copytree(SOURCE, TRAIN_SOURCE)
    record_time('copy_training_source', started)
    for name in ('triton_cache', 'torch_extensions'):
        (RUN_DIR / name).mkdir(exist_ok=True)
    short_tmp = Path('/tmp') / ('rtw_tr_' + hashlib.sha256(str(RUN_DIR).encode()).hexdigest()[:10])
    short_tmp.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    for key in ('ROCR_VISIBLE_DEVICES', 'CUDA_VISIBLE_DEVICES', 'GPU_DEVICE_ORDINAL'):
        env.pop(key, None)
    env.update({
        'HIP_VISIBLE_DEVICES': str(GPU_ID), 'AITER_USE_SYSTEM_TRITON': '1',
        'AITER_TRITON_ONLY': '1', 'FLASH_ATTENTION_TRITON_AMD_ENABLE': 'TRUE',
        'SETUPTOOLS_SCM_PRETEND_VERSION': '0.0.0',
        'PYTORCH_ALLOC_CONF': 'expandable_segments:True',
        'LORA_SOURCE': str(TRAIN_SOURCE), 'LORA_OUTPUT': str(RUN_DIR),
        'LORA_DATA_MANIFEST': str(RUN_DIR / 'task_only.txt'),
        'LORA_GPU_COUNT': str(len(GPU_IDS)), 'LORA_MICRO_BATCH': str(MICRO_BATCH),
        'LORA_GRAD_ACCUM': str(GRAD_ACCUM), 'LORA_MAX_STEPS': str(STEPS),
        'LORA_SAVE_STEPS': str(STEPS), 'TMPDIR': str(short_tmp),
        'TRITON_CACHE_DIR': str(RUN_DIR / 'triton_cache'),
        'TORCH_EXTENSIONS_DIR': str(RUN_DIR / 'torch_extensions'),
    })
    started = time.monotonic()
    run_logged(['bash', str(SCRIPT_ROOT / 'training/run_lora_task.sh')], log, env=env)
    if not CHECKPOINT.is_dir() or f'Reached max_steps={STEPS}' not in log.read_text(errors='replace'):
        raise RuntimeError(f'LoRA did not reach step {STEPS}; see {log}')
    record_time('train', started)
print('checkpoint:', CHECKPOINT)
print('training seconds:', stage_times.get('train', {}).get('seconds'))


## 5. Extract and merge the adapter

Extract only the LoRA tensors from the distributed checkpoint and merge them into the official base one safetensors shard at a time. The resulting Hugging Face checkpoint is used for trained-policy inference.


In [ ]:
if not ADAPTER.is_file():
    started = time.monotonic()
    run_logged([sys.executable, str(SCRIPT_ROOT / 'checkpoints/extract_lora_adapters_dcp.py'),
                '--model-dir', str(CHECKPOINT), '--output', str(ADAPTER),
                '--rank', '16', '--alpha', '32'], RUN_DIR / 'extract.log')
    record_time('extract_adapter', started)
if not (MERGED / 'model.safetensors.index.json').is_file():
    started = time.monotonic()
    run_logged([sys.executable, str(SCRIPT_ROOT / 'checkpoints/merge_lora_adapters_streaming.py'),
                '--base-model', str(BASE), '--adapters', str(ADAPTER),
                '--output', str(MERGED)], RUN_DIR / 'merge.log')
    record_time('merge_model', started)
if not ADAPTER.is_file() or not (MERGED / 'model.safetensors.index.json').is_file():
    raise RuntimeError('Adapter extraction or model merge is incomplete')
manifest = json.loads((RUN_DIR / 'manifest.json').read_text())
manifest['adapter_sha256'] = hashlib.sha256(ADAPTER.read_bytes()).hexdigest()
save_json(RUN_DIR / 'manifest.json', manifest)
print('adapter:', ADAPTER, 'merged model:', MERGED)
print('extract seconds:', stage_times.get('extract_adapter', {}).get('seconds'))
print('merge seconds:', stage_times.get('merge_model', {}).get('seconds'))


## 6. Evaluate the trained policy

Run the trained policy on the same 10 clean and 10 randomized seeds as the base policy. This main evaluation also runs without video.


In [ ]:
trained_results = evaluate('lora')
print('trained lora:', trained_results)


## 7. Compare success and time

Compare the same 10 seeds before and after LoRA. The complete workflow time includes live base evaluation, training, merging and live trained-policy evaluation.


In [ ]:
summary = json.loads((RUN_DIR / 'evaluations/no_video/summary.json').read_text())
from IPython.display import Markdown, display
table = ['| Scene | Base | After LoRA | Gain |', '|---|---:|---:|---:|']
for scene, suffix in (('clean', 'clean'), ('randomized', 'random')):
    before, after = summary[f'baseline_{suffix}'], summary[f'lora_{suffix}']
    table.append(f'| {scene} | {before}/{len(SEEDS)} | {after}/{len(SEEDS)} | '
                 f'{after - before:+d}/{len(SEEDS)} |')
display(Markdown(chr(10).join(table)))
for scene, suffix in (('clean', 'clean'), ('randomized', 'random')):
    base_file = RUN_DIR / 'evaluations/no_video' / f'baseline_{suffix}_episodes.jsonl'
    tuned_file = RUN_DIR / 'evaluations/no_video' / f'lora_{suffix}_episodes.jsonl'
    base_rows = [json.loads(line) for line in base_file.read_text().splitlines() if line.strip()]
    tuned_rows = [json.loads(line) for line in tuned_file.read_text().splitlines() if line.strip()]
    if ([int(row['seed']) for row in base_rows] != SEEDS or
            [int(row['seed']) for row in tuned_rows] != SEEDS or
            any(row.get('error') is not None for row in base_rows + tuned_rows)):
        raise RuntimeError(f'Incomplete paired {scene} evaluation')
    details = [f'### {scene}: 10 paired episodes', '',
               '| Seed | Base policy | After LoRA |',
               '|---:|:---:|:---:|']
    for base_row, tuned_row in zip(base_rows, tuned_rows):
        before = 'Success' if base_row['success'] else 'Failure'
        after = 'Success' if tuned_row['success'] else 'Failure'
        details.append(f"| {base_row['seed']} | {before} | {after} |")
    display(Markdown(chr(10).join(details)))
common = sum(stage_times.get(name, {}).get('seconds', 0) for name in
             ('preflight', 'prepare_evaluation', 'copy_training_source',
              'train', 'extract_adapter', 'merge_model'))
evaluation_seconds = sum(stage_times[f'evaluate_{method}_no_video']['seconds']
                 for method in ('baseline', 'lora'))
print('evaluation:', evaluation_seconds, 's; complete workflow:', common + evaluation_seconds, 's')
print('complete workflow:', round((common + evaluation_seconds) / 60, 1), 'minutes')
record_time('notebook_wall', notebook_started)


## 8. Optional paired video example

Video recording and display are off by default. If you want a visual example, set SHOW_OPTIONAL_VIDEOS to True in the next cell and run that cell after the result tables. It selects the first seed whose base policy failed and trained policy succeeded, checking clean first and then randomized. Only those two episodes are rerun with video. If no such seed exists, the notebook says so.


In [ ]:
from IPython.display import Markdown, Video, display
from video_files import find_video

SHOW_OPTIONAL_VIDEOS = False  # Set True, then run only this cell to record two examples.
if not SHOW_OPTIONAL_VIDEOS:
    display(Markdown('Optional video showcase is off. Set SHOW_OPTIONAL_VIDEOS to True to record a paired example.'))
else:
    started = time.monotonic()
    selected = None
    for scene, suffix in (('clean', 'clean'), ('randomized', 'random')):
        base_path = RUN_DIR / 'evaluations/no_video' / f'baseline_{suffix}_episodes.jsonl'
        trained_path = RUN_DIR / 'evaluations/no_video' / f'lora_{suffix}_episodes.jsonl'
        base_rows = [json.loads(line) for line in base_path.read_text().splitlines() if line.strip()]
        trained_rows = [json.loads(line) for line in trained_path.read_text().splitlines() if line.strip()]
        if ([int(row['seed']) for row in base_rows] != SEEDS or
                [int(row['seed']) for row in trained_rows] != SEEDS):
            raise RuntimeError(f'Missing paired 10-seed {scene} results')
        selected = next((int(before['seed']) for before, after in zip(base_rows, trained_rows)
                         if not before['success'] and after['success']), None)
        if selected is not None:
            break
    if selected is None:
        display(Markdown('No paired seed changed from baseline failure to trained success in these 10 episodes.'))
    else:
        tag = f'video_showcase_{suffix}_seed{selected}'
        showcase = RUN_DIR / 'evaluations' / tag
        recorded_rows = []
        for offset, policy in enumerate(('baseline', 'lora')):
            command = [sys.executable, str(SCRIPT_ROOT / 'eval/run_paired_eval.py'),
                       '--run-dir', str(RUN_DIR), '--result-tag', tag,
                       '--hip-gpu', str(EVAL_GPU), '--port', str(PORT + 10 + offset),
                       '--count', '1', '--steps', str(STEPS), '--task-name', TASK_NAME,
                       '--method', policy, '--scene', suffix, '--seed', str(selected),
                       '--video']
            run_logged(command, RUN_DIR / f'showcase_{policy}_{suffix}_seed{selected}.log')
            path = showcase / f'{policy}_{suffix}_episodes.jsonl'
            rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
            if len(rows) != 1 or int(rows[0]['seed']) != selected or rows[0].get('error') is not None:
                raise RuntimeError(f'Invalid optional video result: {path}')
            recorded_rows.append(rows[0])
        before, after = recorded_rows
        if before['success'] or not after['success']:
            display(Markdown(f'Video rerun for {scene} seed {selected} changed outcome; paired failure/success videos are not shown.'))
        else:
            for policy, row in zip(('baseline', 'lora'), recorded_rows):
                outcome = 'Failure' if policy == 'baseline' else 'Success'
                video = find_video(showcase / 'rollouts' / f'{policy}_{suffix}', row)
                display(Markdown(f'### {scene} seed {selected}: {policy} {outcome}'))
                display(Video(filename=str(video), embed=True, mimetype='video/mp4'))
    record_time('optional_video_showcase', started)
